# 05 · CP2: model build (Milestone 3; spec section 9.1)

Runs on an **L4** (about 1–1.7 compute units). It needs the shards of notebook 04. Checks:

1. model summary and parameter counts (M2 trainable < 5 M);
2. output shapes (spec section 8) at 256 × 256;
3. frozen CLIP weights unchanged after training steps (SHA-256 before and after);
4. **64-image overfit**: ≥ 99% train accuracy within 300 steps, augmentation and dropout off;
5. **kill and resume** in a subprocess: the resumed run continues at the checkpointed step and
   its losses stay within 1% of an uninterrupted run;
6. timing: views < 5 ms and trace branch < 15 ms per image, loader ≥ 150 images/s, batch 32 at
   256² in bf16 fits (peak memory), with and without channels_last;
7. a short validation pass (≤ 0.5 CU).

The CLIP weights (commit `32bd6428…`, pinned in `configs/base.yaml`) are downloaded into the
local disk's `/content/hf_cache` each session (about 1.7 GB, under a minute): Google Drive cannot
hold Hugging Face's cache, whose files are symbolic links.
Before loading, the weights file's SHA-256 is checked against `model.semantic.weights_sha256`
(`a2bf730a…`); a different file stops the run with its size and hash.

The shards are first copied to the local disk (`/content/data/shards`, about 16 GB, a few minutes;
checked shard by shard): reads through Colab's Drive mount cost about a quarter of a second each.

**Result** (send to Claude): `Paper 1/runs/checkpoints/cp2_model_build.json`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings, config, pinned CLIP revision, shards
import json, logging, os, subprocess, sys, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import torch
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info

CU_PER_HOUR = 1.7          # read the live rate in Colab's resource panel and update
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake shards, tiny CLIP, CPU
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp2_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
# Hugging Face's cache links snapshot files to blobs with symbolic links, which Google Drive cannot
# store (it saved the 79-byte link text as model.safetensors): keep the cache on the local disk.
# The pinned revision and the frozen-CLIP SHA-256 in every checkpoint keep the weights identical.
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
if not DRY:
    assert torch.cuda.is_available(), 'CP2 needs a GPU runtime (L4)'
    cfg0 = load_config(CONFIG)
    REVISION = cfg0.model.semantic.revision
    if not REVISION:   # resolve the current commit of the CLIP repository once and pin it
        from huggingface_hub import model_info
        REVISION = model_info(cfg0.model.semantic.backbone).sha
    OVERRIDES.append(f'model.semantic.revision={REVISION}')
else:
    REVISION = 'tiny'
cfg = load_config(CONFIG, OVERRIDES)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| CLIP revision', REVISION)

In [ ]:
# Cell 3: the shards (dry run: fake world -> fake CP1 run -> pack)
SHARDS = paths.shards
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    out = paths.runs / 'checkpoints' / 'cp1'
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=out, cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(out / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
done = pack.read_done(SHARDS)
assert done, f'no finished shards in {SHARDS}: run notebook 04 first'
print('shards:', done['images'], 'images |', done['mask_rule'])
# read through a local copy: each read through Colab's Drive mount costs ~0.25 s (15 Nov)
LOCAL_SHARDS = paths.local_data / 'shards'          # /content/data/shards on Colab
staged = pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print)
print('staged:', staged)
SHARDS = LOCAL_SHARDS
RESULT = {'step': 'CP2 model build', 'dry_run': DRY, 'branch': BRANCH, 'session': info,
          'config': CONFIG, 'overrides': OVERRIDES, 'clip_revision': REVISION,
          'clip_weights_sha256': cfg.model.semantic.get('weights_sha256'),
          'shards_staged': staged, 'checks': {}}

In [ ]:
# Cell 4: model summary, shapes, M2 trainable count
from eviground.train import checkpoint as ckpt
from eviground.train.trainer import Trainer

RUNS = paths.runs / 'cp2'
tr = Trainer(cfg, SHARDS, RUNS / 'overfit', resume=None)
m = tr.model
count = lambda mod: sum(p.numel() for p in mod.parameters() if p.requires_grad)
params = {
    'total': sum(p.numel() for p in m.parameters()), 'trainable': count(m),
    'm2_semantic_trainable': count(m.semantic), 'm2_lora': sum(p.numel() for p in m.semantic.lora_parameters()),
    'm3_trace': count(m.trace), 'm4_fusion': count(m.fusion),
    'heads': count(m.cls_head) + count(m.decoder),
}
crop = int(cfg.data.crop)
x = torch.rand(2, 3, crop, crop, device=tr.device)
m.eval()
with torch.no_grad(), torch.autocast(tr.device.type, dtype=torch.bfloat16, enabled=not DRY):
    o = m(x)
shapes = {k: list(v.shape) for k, v in o.items()}
want = {'logits': [2, 3], 'mask_logits': [2, 1, crop, crop], 'gate': [2, 1]}
RESULT['checks']['params'] = {**params, 'pass': params['m2_semantic_trainable'] < 5_000_000}
RESULT['checks']['shapes'] = {'got': shapes, 'pass': all(shapes[k] == v for k, v in want.items())
                              and shapes['alpha'][1] == shapes['z'][1]}
from collections import Counter
pools = Counter(f"{r['label']}/{r['source']}" for r in tr.train_ds.rows)
RESULT['data'] = {
    'train_pools': dict(sorted(pools.items())),
    'left_out_never_valid': len(tr.train_ds.dropped),
    'train_items_per_epoch': len(tr.train_ds), 'val_items': len(tr.val_ds),
    'val_left_out': len(tr.val_ds.dropped),
}
print(json.dumps(params, indent=1)); print(shapes); print(RESULT['data'])

In [ ]:
# Cell 5: 64-image overfit (augmentation and dropout off), frozen CLIP unchanged afterwards
sha_before = tr.clip_sha256
t = time.time()
of = tr.overfit()
sha_after = ckpt.frozen_clip_sha256(tr.model)
RESULT['checks']['frozen_clip_unchanged'] = {'sha256': sha_before, 'pass': sha_after == sha_before}
RESULT['checks']['overfit_64'] = {**of, 'minutes': round((time.time() - t) / 60, 2),
                                  'pass': of['reached_at_step'] is not None}
print({k: v for k, v in of.items() if k != 'history'}, '| frozen unchanged:', sha_after == sha_before)
del tr, m
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Cell 6: kill and resume in a subprocess (checkpoint every 20 steps, killed after step 30)
import csv
N, KILL, EVERY = (6, 3, 2) if DRY else (40, 30, 20)
base = [sys.executable, os.path.join(REPO, 'scripts', 'train.py'), '--config', CONFIG,
        '--shards', str(SHARDS), '--max-steps', str(N)]
over = OVERRIDES + [f'checkpoint.every_steps={EVERY}']
import shutil
ref, cut = RUNS / 'resume_ref', RUNS / 'resume_cut'
for d in (ref, cut):   # this notebook's own scratch runs
    shutil.rmtree(d, ignore_errors=True)
runs = {
    'reference': subprocess.run(base + ['--run-dir', str(ref)] + over),
    'killed': subprocess.run(base + ['--run-dir', str(cut), '--kill-at-step', str(KILL)] + over),
    'resumed': subprocess.run(base + ['--run-dir', str(cut)] + over),
}
def steps(d):
    rows = {}
    with open(d / 'steps.csv', newline='') as fh:
        for r in csv.DictReader(fh):
            rows[int(r['global_step'])] = r   # the resumed run's rows replace the killed run's
    return rows
a, b = steps(ref), steps(cut)
compare = range(EVERY + 1, N + 1)
rel = [abs(float(a[s]['loss_total']) - float(b[s]['loss_total'])) / abs(float(a[s]['loss_total']))
       for s in compare]
same_uids = all(a[s]['first_uid'] == b[s]['first_uid'] for s in compare)
last = ckpt.load(cut / 'last.pt')['run']
RESULT['checks']['kill_resume'] = {
    'returncodes': {k: v.returncode for k, v in runs.items()},
    'killed_after_step': KILL, 'resumed_from_step': EVERY, 'final_step': last['global_step'],
    'max_rel_loss_diff': max(rel), 'mean_rel_loss_diff': sum(rel) / len(rel),
    'same_batches': same_uids,
    'pass': runs['killed'].returncode == 137 and runs['resumed'].returncode == 0
            and last['global_step'] == N and same_uids and max(rel) <= 0.01,
}
print(RESULT['checks']['kill_resume'])

In [ ]:
# Cell 7: timing and memory (views, trace, loader, a training step at batch 32), +/- channels_last
from torch.utils.data import DataLoader
from eviground.models.eviground import build_model
from eviground.train.trainer import _collate
from eviground.data.datasets import ForensicDataset

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
B = 4 if DRY else 32
def timed(fn, n=5 if DRY else 20):
    for _ in range(2): fn()
    torch.cuda.synchronize() if dev.type == 'cuda' else None
    t = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize() if dev.type == 'cuda' else None
    return (time.perf_counter() - t) / n

timing = {}
for cl in (False, True):
    c = load_config(CONFIG, OVERRIDES + [f'model.trace.channels_last={cl}'])
    torch.manual_seed(0)
    mm = build_model(c.model).to(dev)
    x = torch.rand(B, 3, crop, crop, device=dev)
    if cl: x = x.contiguous(memory_format=torch.channels_last)
    ac = lambda: torch.autocast(dev.type, dtype=torch.bfloat16, enabled=dev.type == 'cuda')
    with torch.no_grad(), ac():
        v = mm.views(x)
        t_views = timed(lambda: mm.views(x))
        tr_in = v['trace'].contiguous(memory_format=torch.channels_last) if cl else v['trace']
        t_trace = timed(lambda: mm.trace(tr_in))
    opt = torch.optim.AdamW([p for p in mm.parameters() if p.requires_grad], lr=1e-4)
    torch.cuda.reset_peak_memory_stats() if dev.type == 'cuda' else None
    def step():
        with ac():
            out = mm(x)
        loss = out['logits'].float().logsumexp(-1).mean() + out['mask_logits'].float().mean()
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    t_step = timed(step, n=3 if DRY else 10)
    peak = torch.cuda.max_memory_allocated() / 2**30 if dev.type == 'cuda' else None
    timing[f'channels_last={cl}'] = {
        'views_ms_per_image': 1000 * t_views / B, 'trace_ms_per_image': 1000 * t_trace / B,
        'train_step_s_batch': t_step, 'peak_mem_gb_batch': peak, 'batch': B,
    }
    del mm, opt
    torch.cuda.empty_cache() if dev.type == 'cuda' else None
ds = ForensicDataset(SHARDS, 'train', crop=crop, n_per_class=64 if DRY else 2000, seed=42)
nw = int(cfg.data.num_workers)
dl = DataLoader(ds, batch_size=B, num_workers=nw, collate_fn=_collate, persistent_workers=False)
t, n = time.perf_counter(), 0
for i, b in enumerate(dl):
    n += len(b['uid'])
    if i == (10 if DRY else 60): break
timing['loader_images_per_s'] = n / (time.perf_counter() - t)
timing['loader_workers'] = nw
best = min(timing[k]['trace_ms_per_image'] for k in ('channels_last=False', 'channels_last=True'))
RESULT['checks']['timing'] = {
    **timing,
    'channels_last_faster': timing['channels_last=True']['train_step_s_batch']
                            < timing['channels_last=False']['train_step_s_batch'],
    'pass': (min(timing[k]['views_ms_per_image'] for k in ('channels_last=False', 'channels_last=True')) < 5
             and best < 15 and timing['loader_images_per_s'] >= 150
             and all(timing[k]['peak_mem_gb_batch'] is not None for k in ('channels_last=False', 'channels_last=True'))),
}
print(json.dumps(RESULT['checks']['timing'], indent=1))

In [ ]:
# Cell 8: a short validation pass with the overfit model's weights discarded (fresh model)
vt = Trainer(cfg, SHARDS, RUNS / 'validation', resume=None)
t = time.time()
val = vt.validate()
RESULT['checks']['validation_pass'] = {
    'images': len(vt.val_ds), 'minutes': round((time.time() - t) / 60, 2),
    **{k: v for k, v in val.items() if k != 'per_source'},
    'pass': len(vt.val_ds) > 0,
}
print({k: v for k, v in RESULT['checks']['validation_pass'].items()})
del vt

In [ ]:
# Cell 9: verdict of the script (Dhanush decides the checkpoint) and the result file
import datetime
hours = (time.time() - T0) / 3600
RESULT.update({
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR,
    'compute_units': round(hours * CU_PER_HOUR, 2),
    'shards': {k: done.get(k) for k in ('images', 'manifest_sha256', 'mask_rule')},
    'index_sha256': pack.file_sha256(SHARDS / 'index.parquet'),
})
RESULT['verdict'] = 'PASS' if all(c['pass'] for c in RESULT['checks'].values()) else 'FAIL'
out = paths.runs / 'checkpoints' / 'cp2_model_build.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(RESULT, indent=1, default=str))
print({k: v['pass'] for k, v in RESULT['checks'].items()})
print('script verdict:', RESULT['verdict'], '| CU', RESULT['compute_units'], '| saved', out)